# Local projections: incertidumbre comercial y actividad manufacturera por rama

Lee la base integrada (`data/processed/base_incertidumbre_comercial_mexico.xlsx`), construye **variaciones a 12 meses** de todas las variables, construye la **sorpresa del TPU** y estima *local projections* (LP) en panel para **producción real, empleo y horas trabajadas**, con y sin el periodo de la pandemia.

**Dos especificaciones** (para h = 0, …, 12 meses; i = rama, t = mes):

**A. Exposición diferencial** (efectos fijos de rama y de mes):

Δ12 y(i,t+h) = a_i + d_t + β_h · choque_t × E_i + Σ_{l=1..3} [ρ'_l Δ12 Y(i,t−l) + θ_l choque_{t−l} × E_i] + Σ_{l=0..3} [γ_l Δ12 arancel(i,t−l) + κ_l Δ12 IP_EEUU_{t−l} × E_i] + e

**B. Efecto promedio** (efectos fijos de rama, sin efectos de mes):

Δ12 y(i,t+h) = a_i + β_h · choque_t + Σ_{l=1..3} [ρ'_l Δ12 Y(i,t−l) + θ_l choque_{t−l} + φ_l Δ12 TC_{t−l} + ψ_l Δ12 VIX_{t−l}] + Σ_{l=0..3} [γ_l Δ12 arancel(i,t−l) + κ_l Δ12 IP_EEUU_{t−l}] + e

- **y** ∈ {producción real, empleo, horas}. **Y** = el vector de las **tres**: los rezagos de las tres variables entran como control en todas las regresiones.
- **E_i** = coeficiente de exportación de la MIP 2018 (decisión D9).
- **IP_EEUU** = producción industrial total de EE.UU. (INDPRO). En A entra multiplicada por la exposición (las ramas exportadoras dependen más de la demanda de EE.UU.); en B, directamente. Entra también en el mes t, porque es el factor de demanda que se quiere separar.
- **Tipo de cambio y VIX** (solo en B; en A los absorben los efectos de mes) entran solo rezagados, porque reaccionan en el mismo mes a las noticias comerciales.
- **Choques:** (1) la **sorpresa del TPU** (principal) y (2) la variación anual del log del TPU (comparación). Ambos estandarizados: 1 = una desviación estándar.
- **Errores estándar:** Driscoll-Kraay con rezago máximo 12 + h.
- **Muestras:** completa y **sin pandemia** (se excluyen las observaciones con t o t+h entre 2020-03 y 2021-06).

El notebook **no** modifica la base integrada.

In [1]:
import html, platform, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore")
def _raiz():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "data" / "processed").is_dir() and (q / "docs").is_dir():
            return q
    raise RuntimeError("No se encontró la raíz del repositorio.")
RAIZ = _raiz()
display(HTML("""<style>
.tabla{font-size:82%;border-collapse:collapse;margin-bottom:8px}.tabla th{background:#dce6f1;border:1px solid #bbb;padding:3px 6px;text-align:left}
.tabla td{border:1px solid #ddd;padding:2px 6px}.irf-grid{display:grid;grid-template-columns:repeat(2,minmax(0,430px));gap:10px 16px}
</style>"""))

def tabla(df, titulo=None, max_filas=120, dec=3):
    if titulo:
        display(Markdown(f"**{titulo}**"))
    d = df.head(max_filas).copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda v: "NA" if pd.isna(v) else f"{v:,.{dec}f}")
    display(HTML(d.to_html(index=False, border=0, classes="tabla", escape=True)))

def aviso(texto, tipo="info"):
    c = {"info": ("#e8f1fb", "#2c5aa0"), "revisar": ("#fff4e0", "#b26a00"), "ok": ("#e7f5ec", "#1b7f3b")}[tipo]
    display(HTML(f'<div style="background:{c[0]};border-left:6px solid {c[1]};padding:8px 12px;margin:8px 0">{texto}</div>'))

base = pd.read_excel(RAIZ / "data/processed/base_incertidumbre_comercial_mexico.xlsx", sheet_name="base_integrada", dtype={"rama": str, "subsector": str})
base["mes"] = pd.to_datetime(base.mes)
assert not base.duplicated(["rama", "mes"]).any()
assert "eeuu_ip_total" in base, "La base no tiene eeuu_ip_total: vuelva a ejecutar 01_preparacion_base_integrada.ipynb"
display(Markdown(f"**Base:** {base.shape[0]:,} filas, {base.rama.nunique()} ramas, {base.mes.min():%Y-%m} a {base.mes.max():%Y-%m}."))

**Base:** 8,858 filas, 86 ramas, 2018-01 a 2026-07.

## 1. Variaciones a 12 meses
Diferencia de logaritmos para variables positivas; diferencia simple (en puntos) para tasas o índices que pueden valer cero. Se compara con el mismo mes del año anterior por fecha, no por posición.

In [2]:
base = base.sort_values(["rama", "mes"]).reset_index(drop=True)
transformaciones = [
    ("emim_valor_produccion_real", "d12_produccion", "log", "Producción real (EMIM)"),
    ("emim_personal_ocupado", "d12_empleo", "log", "Personal ocupado (EMIM)"),
    ("emim_horas_trabajadas", "d12_horas", "log", "Horas trabajadas (EMIM)"),
    ("tpu", "d12_tpu", "log", "TPU (Caldara et al.)"),
    ("eeuu_ip_total", "d12_ip_eeuu", "log", "Producción industrial total de EE.UU."),
    ("eeuu_imp_mx_tasa_efectiva", "d12_arancel", "dif_pp", "Tasa arancelaria efectiva de EE.UU. a la rama"),
    ("tc_fix", "d12_tc", "log", "Tipo de cambio FIX"),
    ("vix_promedio_mensual", "d12_vix", "log", "VIX (promedio mensual)"),
    ("wtui_usa", "d12_wtui_usa", "dif", "WTUI EE.UU."),
    ("wtui_mundo_pib", "d12_wtui_mundo", "dif", "WTUI mundo"),
    ("wtui_mex", "d12_wtui_mex", "dif", "WTUI México"),
    ("epu_mex", "d12_epu", "log", "EPU México"),
]
anterior = base[["rama", "mes"]].assign(mes=base.mes + pd.DateOffset(years=1))
filas = []
for orig, nueva, metodo, desc in transformaciones:
    x = base[orig].astype(float)
    if metodo == "log":
        assert (x.dropna() > 0).all(), orig
        x = np.log(x)
    elif metodo == "dif_pp":
        x = 100 * x
    base = base.merge(anterior.assign(previo=x.values), on=["rama", "mes"], how="left", validate="one_to_one")
    base[nueva] = x.values - base.pop("previo")
    filas.append(dict(variable=nueva, original=orig, descripcion=desc,
                      metodo={"log": "Δ log", "dif": "diferencia en puntos del índice", "dif_pp": "diferencia en puntos porcentuales"}[metodo],
                      observaciones=int(base[nueva].notna().sum()), primer_mes=base.mes[base[nueva].notna()].min().strftime("%Y-%m")))
tabla(pd.DataFrame(filas), "Variables transformadas")
E = base.groupby("rama").mip_coef_exportacion_2018.first()
E_P25, E_P75 = E.quantile(0.25), E.quantile(0.75)
ESC_A = 100 * (E_P75 - E_P25)
display(Markdown(f"**Exposición (MIP 2018):** p25 = {E_P25:.3f}, p75 = {E_P75:.3f}. En la especificación A los efectos se reportan para la diferencia p75 − p25."))

**Variables transformadas**

variable,original,descripcion,metodo,observaciones,primer_mes
d12_produccion,emim_valor_produccion_real,Producción real (EMIM),Δ log,7786,2019-01
d12_empleo,emim_personal_ocupado,Personal ocupado (EMIM),Δ log,7826,2019-01
d12_horas,emim_horas_trabajadas,Horas trabajadas (EMIM),Δ log,7826,2019-01
d12_tpu,tpu,TPU (Caldara et al.),Δ log,7826,2019-01
d12_ip_eeuu,eeuu_ip_total,Producción industrial total de EE.UU.,Δ log,7826,2019-01
d12_arancel,eeuu_imp_mx_tasa_efectiva,Tasa arancelaria efectiva de EE.UU. a la rama,diferencia en puntos porcentuales,7735,2019-01
d12_tc,tc_fix,Tipo de cambio FIX,Δ log,7826,2019-01
d12_vix,vix_promedio_mensual,VIX (promedio mensual),Δ log,7826,2019-01
d12_wtui_usa,wtui_usa,WTUI EE.UU.,diferencia en puntos del índice,7826,2019-01
d12_wtui_mundo,wtui_mundo_pib,WTUI mundo,diferencia en puntos del índice,7826,2019-01


**Exposición (MIP 2018):** p25 = 0.158, p75 = 0.618. En la especificación A los efectos se reportan para la diferencia p75 − p25.

## 2. La sorpresa del TPU
Regresión de pronóstico con toda la historia disponible (1990-2026): log TPU_t sobre 12 rezagos propios, 3 rezagos del log del VIX y **3 rezagos del log de la producción industrial de EE.UU.** La sorpresa es el residuo: la parte del TPU que no se anticipaba con la información del mes anterior. Robustez: error de pronóstico fuera de muestra (ventana creciente desde 2010).

In [3]:
tpu_raw = pd.read_excel(RAIZ / "data/raw/incertidumbre/tpu_caldara_iacoviello.xlsx", sheet_name="TPU_MONTHLY")
tpu_h = tpu_raw.assign(mes=pd.to_datetime(tpu_raw.DATE).dt.to_period("M").dt.to_timestamp()).set_index("mes").TPU.astype(float)
vix_raw = pd.read_csv(RAIZ / "data/raw/controles_macroeconomicos/fred_vix_diario.csv", na_values=["", "."], parse_dates=["observation_date"])
vix_h = vix_raw.dropna().set_index("observation_date").VIXCLS.resample("MS").mean()
ip_raw = pd.read_csv(RAIZ / "data/raw/produccion_estados_unidos/fred_indpro_mensual.csv", parse_dates=["observation_date"])
ip_h = ip_raw.set_index("observation_date").INDPRO.astype(float)
serie_mes = base.drop_duplicates("mes").set_index("mes")
comun = serie_mes[["tpu", "vix_promedio_mensual", "eeuu_ip_total"]].join(pd.DataFrame({"a": tpu_h, "b": vix_h, "c": ip_h}), how="inner")
assert np.allclose(comun.tpu, comun.a) and np.allclose(comun.vix_promedio_mensual, comun.b) and np.allclose(comun.eeuu_ip_total, comun.c)
aviso(f"Las series originales del TPU, VIX y producción industrial de EE.UU. coinciden con la base en los {len(comun)} meses comunes.", "ok")

pron = pd.DataFrame({"ltpu": np.log(tpu_h), "lvix": np.log(vix_h), "lip": np.log(ip_h)})
X_cols = []
for v, n in [("ltpu", 12), ("lvix", 3), ("lip", 3)]:
    for l in range(1, n + 1):
        pron[f"{v}_l{l}"] = pron[v].shift(l)
        X_cols.append(f"{v}_l{l}")
pron = pron.dropna(subset=["ltpu", *X_cols])
reg = sm.OLS(pron.ltpu, sm.add_constant(pron[X_cols])).fit()
sorpresa = reg.resid
oos = {}
for m in pron.index[pron.index >= "2010-01-01"]:
    previo = pron[pron.index < m]
    r_ = sm.OLS(previo.ltpu, sm.add_constant(previo[X_cols])).fit()
    oos[m] = pron.loc[m, "ltpu"] - float(r_.predict(sm.add_constant(pron.loc[[m], X_cols], has_constant="add")).iloc[0])
sorpresa_oos = pd.Series(oos)
tabla(pd.DataFrame([
    dict(prueba="Meses en la regresión de pronóstico", valor=f"{len(pron)} ({pron.index.min():%Y-%m} a {pron.index.max():%Y-%m})"),
    dict(prueba="R²", valor=f"{reg.rsquared:.3f}"),
    dict(prueba="Prueba F: los rezagos de la producción industrial de EE.UU. no ayudan a pronosticar (valor p)",
         valor=f"{reg.f_test(', '.join(f'lip_l{l} = 0' for l in range(1, 4))).pvalue:.3f}"),
    dict(prueba="Autocorrelación de la sorpresa (rezagos 1, 2, 3, 12)", valor=", ".join(f"{sorpresa.autocorr(k):.2f}" for k in (1, 2, 3, 12))),
    dict(prueba="Correlación con la versión fuera de muestra (2018-2026)", valor=f"{sorpresa['2018':].corr(sorpresa_oos['2018':]):.3f}"),
]), "Diagnóstico de la sorpresa")
top = pd.concat([sorpresa.sort_values(ascending=False).head(8), sorpresa.sort_values().head(4)])
tabla(pd.DataFrame({"mes": [m.strftime("%Y-%m") for m in top.index], "sorpresa_log": top.values, "tipo": ["alza"] * 8 + ["baja"] * 4}),
      "Meses con mayor sorpresa (toda la historia)", dec=2)

base = base.merge(pd.DataFrame({"sorpresa_tpu": sorpresa, "sorpresa_tpu_oos": sorpresa_oos}).rename_axis("mes").reset_index(), on="mes", how="left", validate="many_to_one")
serie_mes = base.drop_duplicates("mes").set_index("mes")
ventana = serie_mes.loc[serie_mes.index >= "2018-01-01"]
ESCALA = {}
for c in ["sorpresa_tpu", "sorpresa_tpu_oos", "d12_tpu", "d12_wtui_usa", "d12_wtui_mundo", "d12_wtui_mex", "d12_epu"]:
    ESCALA[c] = ventana[c].std()
    base[c + "_std"] = base[c] / ESCALA[c]
serie_mes = base.drop_duplicates("mes").set_index("mes")
tabla(pd.DataFrame([dict(choque=k, desviacion_estandar_2018_2026=v) for k, v in ESCALA.items()]), "Desviación estándar usada para estandarizar cada choque")

def svg_linea(x, y, titulo, ylab, ancho=860, alto=220, sombra=None):
    L, R, T, B = 60, 20, 30, 34
    lo, hi = np.nanmin(y), np.nanmax(y)
    X = lambda i: L + i / (len(x) - 1) * (ancho - L - R)
    Y = lambda v: T + (hi - v) / (hi - lo) * (alto - T - B)
    p = [f'<svg viewBox="0 0 {ancho} {alto}" style="width:100%;max-width:{ancho}px;font:11px system-ui,sans-serif" role="img">',
         f'<text x="{L}" y="16" style="font-size:13px;font-weight:600">{html.escape(titulo)}</text>']
    if sombra:
        p.append(f'<rect x="{X(sombra[0]):.1f}" y="{T}" width="{X(sombra[1]) - X(sombra[0]):.1f}" height="{alto - T - B}" fill="#e6e5e1"/>')
    for t in np.linspace(lo, hi, 5):
        p.append(f'<line x1="{L}" x2="{ancho - R}" y1="{Y(t):.1f}" y2="{Y(t):.1f}" stroke="#eeeeec"/><text x="{L - 6}" y="{Y(t) + 4:.1f}" text-anchor="end" fill="#7a7974">{t:.1f}</text>')
    p.append(f'<line x1="{L}" x2="{ancho - R}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="#52514e"/>')
    p.append(f'<polyline points="{" ".join(f"{X(i):.1f},{Y(v):.1f}" for i, v in enumerate(y))}" fill="none" stroke="#2a78d6" stroke-width="2"/>')
    for i, (xx, v) in enumerate(zip(x, y)):
        p.append(f'<circle cx="{X(i):.1f}" cy="{Y(v):.1f}" r="6" fill="transparent"><title>{xx}: {v:.2f}</title></circle>')
        if xx.endswith("-01"):
            p.append(f'<text x="{X(i):.1f}" y="{alto - B + 15}" text-anchor="middle" fill="#7a7974">{xx[:4]}</text>')
    p.append(f'<text x="14" y="{T + (alto - T - B) / 2}" transform="rotate(-90 14 {T + (alto - T - B) / 2})" text-anchor="middle" fill="#7a7974">{ylab}</text></svg>')
    return "".join(p)

PANDEMIA = (pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-01"))
s2 = serie_mes.sorpresa_tpu_std.loc["2018-01-01":].dropna()
ip2 = [i for i, m in enumerate(s2.index) if PANDEMIA[0] <= m <= PANDEMIA[1]]
display(HTML(svg_linea([m.strftime("%Y-%m") for m in s2.index], s2.values, "Sorpresa del TPU, estandarizada, 2018-2026 (gris: pandemia)",
                       "desv. estándar", sombra=(ip2[0], ip2[-1]))))

**Diagnóstico de la sorpresa**

prueba,valor
Meses en la regresión de pronóstico,437 (1990-04 a 2026-08)
R²,0.882
Prueba F: los rezagos de la producción industrial de EE.UU. no ayudan a pronosticar (valor p),0.790
"Autocorrelación de la sorpresa (rezagos 1, 2, 3, 12)","-0.00, -0.00, -0.00, -0.00"
Correlación con la versión fuera de muestra (2018-2026),0.969


**Meses con mayor sorpresa (toda la historia)**

mes,sorpresa_log,tipo
2018-03,1.43,alza
2024-11,1.06,alza
2025-04,0.84,alza
2016-11,0.81,alza
2013-05,0.69,alza
2019-05,0.65,alza
2015-05,0.63,alza
1993-11,0.62,alza
2013-04,-1.00,baja
2021-02,-0.74,baja


**Desviación estándar usada para estandarizar cada choque**

choque,desviacion_estandar_2018_2026
sorpresa_tpu,0.320
sorpresa_tpu_oos,0.361
d12_tpu,0.855
d12_wtui_usa,0.263
d12_wtui_mundo,"13,192.906"
d12_wtui_mex,0.283
d12_epu,0.695


## 3. Estimador
Los efectos fijos se eliminan por proyecciones alternadas (panel no balanceado por faltantes) y se estima por MCO; errores Driscoll-Kraay (`nw-groupsum` por mes, rezago máximo 12 + h).

In [4]:
Y_VARS = {"produccion": "d12_produccion", "empleo": "d12_empleo", "horas": "d12_horas"}
NOMBRE_Y = {"produccion": "Producción real", "empleo": "Empleo", "horas": "Horas trabajadas"}

def demean_two_way(M, g1, g2, tol=1e-10, max_iter=500):
    M = M.copy()
    for _ in range(max_iter):
        antes = M.copy()
        M = M - M.groupby(g1).transform("mean")
        M = M - M.groupby(g2).transform("mean")
        if np.abs(M.values - antes.values).max() < tol:
            break
    return M

def estimar_lp(y="produccion", choque="sorpresa_tpu_std", espec="A", excluir_pandemia=False, H=12, L=3):
    yv = Y_VARS[y]
    d = base[["rama", "mes", *Y_VARS.values(), choque, "d12_arancel", "d12_ip_eeuu", "d12_tc", "d12_vix", "mip_coef_exportacion_2018"]].copy()
    Ei = d.mip_coef_exportacion_2018 if espec == "A" else 1.0
    d["choque"] = d[choque] * Ei
    d["ip"] = d["d12_ip_eeuu"] * Ei
    g = d.groupby("rama")
    regs = ["choque"]
    for l in range(1, L + 1):
        d[f"choque_l{l}"] = g["choque"].shift(l); regs.append(f"choque_l{l}")
        for v in Y_VARS.values():
            d[f"{v}_l{l}"] = g[v].shift(l); regs.append(f"{v}_l{l}")
    for l in range(0, L + 1):
        d[f"arancel_l{l}"] = g["d12_arancel"].shift(l); regs.append(f"arancel_l{l}")
        d[f"ip_l{l}"] = g["ip"].shift(l); regs.append(f"ip_l{l}")
    if espec == "B":
        for v in ["d12_tc", "d12_vix"]:
            for l in range(1, L + 1):
                d[f"{v}_l{l}"] = g[v].shift(l); regs.append(f"{v}_l{l}")
    filas = []
    for h in range(H + 1):
        d["y_h"] = g[yv].shift(-h)
        d["mes_h"] = g["mes"].shift(-h)
        m = d.dropna(subset=["y_h", *regs])
        if excluir_pandemia:
            dentro = lambda s: (s >= PANDEMIA[0]) & (s <= PANDEMIA[1])
            m = m[~dentro(m.mes) & ~dentro(m.mes_h)]
        cols = ["y_h", *regs]
        Z = demean_two_way(m[cols], m.rama, m.mes) if espec == "A" else m[cols] - m.groupby("rama")[cols].transform("mean")
        t_idx = m.mes.rank(method="dense").astype(int).values
        fit = sm.OLS(Z["y_h"].values, Z[regs].values).fit(cov_type="nw-groupsum", cov_kwds={"time": t_idx, "maxlags": 12 + h})
        filas.append(dict(h=h, beta=fit.params[0], se_dk=fit.bse[0], t_dk=fit.params[0] / fit.bse[0], n=len(m),
                          ramas=m.rama.nunique(), meses=m.mes.nunique(), desde=m.mes.min().strftime("%Y-%m"), hasta=m.mes.max().strftime("%Y-%m")))
    r = pd.DataFrame(filas)
    escala = ESC_A if espec == "A" else 100.0
    r["efecto_pp"] = r.beta * escala
    r["ic90_inf"] = (r.beta - 1.645 * r.se_dk) * escala
    r["ic90_sup"] = (r.beta + 1.645 * r.se_dk) * escala
    return r.assign(variable=y, choque=choque, especificacion=espec, muestra="sin pandemia" if excluir_pandemia else "completa")

def marcas(ymin, ymax, n=5):
    paso = (ymax - ymin) / n
    mag = 10 ** np.floor(np.log10(paso))
    paso = min((k * mag for k in (1, 2, 2.5, 5, 10) if k * mag >= paso), default=10 * mag)
    return np.arange(np.floor(ymin / paso) * paso, ymax + paso / 2, paso), paso

def svg_irf(r, titulo, ancho=430, alto=240, color="#2a78d6"):
    L_, R_, T_, B_ = 52, 14, 34, 34
    b, lo9, hi9 = r.efecto_pp.values, r.ic90_inf.values, r.ic90_sup.values
    esc = r.efecto_pp.values / r.beta.values if (r.beta != 0).all() else np.ones(len(r))
    lo95, hi95 = (r.beta - 1.96 * r.se_dk).values * esc, (r.beta + 1.96 * r.se_dk).values * esc
    ticks, paso = marcas(min(lo95.min(), 0), max(hi95.max(), 0))
    y0, y1 = ticks[0], ticks[-1]
    X = lambda h: L_ + h / 12 * (ancho - L_ - R_)
    Y = lambda v: T_ + (y1 - v) / (y1 - y0) * (alto - T_ - B_)
    banda = lambda lo, hi: " ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(r.h, hi)) + " " + " ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(r.h[::-1], lo[::-1]))
    dec = next(k for k in range(4) if abs(round(paso, k) - paso) < 1e-9)
    p = [f'<svg viewBox="0 0 {ancho} {alto}" style="width:100%;font:11px system-ui,sans-serif" role="img" aria-label="{html.escape(titulo)}">',
         f'<text x="{L_}" y="14" style="font-size:12px;font-weight:600;fill:#0b0b0b">{html.escape(titulo)}</text>']
    for t in ticks:
        p.append(f'<line x1="{L_}" x2="{ancho - R_}" y1="{Y(t):.1f}" y2="{Y(t):.1f}" stroke="#eeeeec"/><text x="{L_ - 6}" y="{Y(t) + 4:.1f}" text-anchor="end" fill="#7a7974">{round(t, dec) + 0.0:.{dec}f}</text>')
    p.append(f'<text x="12" y="{T_ + (alto - T_ - B_) / 2:.0f}" transform="rotate(-90 12 {T_ + (alto - T_ - B_) / 2:.0f})" text-anchor="middle" fill="#7a7974">puntos porcentuales</text>')
    p.append(f'<polygon points="{banda(lo95, hi95)}" fill="{color}" fill-opacity="0.12"/><polygon points="{banda(lo9, hi9)}" fill="{color}" fill-opacity="0.22"/>')
    p.append(f'<line x1="{L_}" x2="{ancho - R_}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="#52514e"/>')
    p.append(f'<polyline points="{" ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(r.h, b))}" fill="none" stroke="{color}" stroke-width="2"/>')
    for h, v, a, c, t in zip(r.h, b, lo9, hi9, r.t_dk):
        p.append(f'<circle cx="{X(h):.1f}" cy="{Y(v):.1f}" r="3" fill="{color}"/><circle cx="{X(h):.1f}" cy="{Y(v):.1f}" r="9" fill="transparent">'
                 f'<title>h={h}: {v:.2f} pp (IC 90%: {a:.2f} a {c:.2f}; t = {t:.1f})</title></circle>')
    for h in range(0, 13, 3):
        p.append(f'<text x="{X(h):.1f}" y="{alto - B_ + 15}" text-anchor="middle" fill="#7a7974">{h}</text>')
    p.append(f'<text x="{(L_ + ancho - R_) / 2}" y="{alto - 4}" text-anchor="middle" fill="#7a7974">meses después del choque (h)</text></svg>')
    return "".join(p)

resultados = []
def bloque(choque, espec, titulo_bloque):
    display(Markdown(f"### {titulo_bloque}"))
    celdas = []
    for y in Y_VARS:
        for sp in (False, True):
            r = estimar_lp(y=y, choque=choque, espec=espec, excluir_pandemia=sp)
            resultados.append(r)
            celdas.append(svg_irf(r, f"{NOMBRE_Y[y]} · {'sin pandemia' if sp else 'muestra completa'}"))
    display(HTML('<div class="irf-grid">' + "".join(celdas) + "</div>"))

## 4. Especificación A: exposición diferencial
Eje vertical: diferencia en puntos porcentuales entre una rama de exposición alta (p75) y una baja (p25) ante 1 desviación estándar del choque. Bandas de 90% (oscura) y 95% (clara). Con la sorpresa (choque de un mes) y la variable dependiente en variación anual, h = 0 a 11 mide el efecto acumulado sobre el **nivel**.

In [5]:
bloque("sorpresa_tpu_std", "A", "A1. Choque: sorpresa del TPU (principal)")

### A1. Choque: sorpresa del TPU (principal)

In [6]:
bloque("d12_tpu_std", "A", "A2. Choque: variación anual del log del TPU (comparación)")

### A2. Choque: variación anual del log del TPU (comparación)

## 5. Especificación B: efecto promedio
Eje vertical: puntos porcentuales por 1 desviación estándar del choque, para la rama promedio. Controla por la producción industrial de EE.UU. (mes t y 3 rezagos) y por tipo de cambio y VIX rezagados.

In [7]:
bloque("sorpresa_tpu_std", "B", "B1. Choque: sorpresa del TPU (principal)")

### B1. Choque: sorpresa del TPU (principal)

In [8]:
bloque("d12_tpu_std", "B", "B2. Choque: variación anual del log del TPU (comparación)")

### B2. Choque: variación anual del log del TPU (comparación)

## 6. Robustez
Especificación A con producción, empleo y horas, sin pandemia: sorpresa fuera de muestra y otras medidas de incertidumbre (en variación anual).

In [9]:
celdas = []
for ch, nombre in [("sorpresa_tpu_oos_std", "Sorpresa fuera de muestra"), ("d12_wtui_usa_std", "WTUI EE.UU."),
                   ("d12_wtui_mundo_std", "WTUI mundo"), ("d12_wtui_mex_std", "WTUI México"), ("d12_epu_std", "EPU México")]:
    for y in Y_VARS:
        r = estimar_lp(y=y, choque=ch, espec="A", excluir_pandemia=True)
        resultados.append(r.assign(robustez=nombre))
        celdas.append(svg_irf(r, f"{nombre} · {NOMBRE_Y[y]}"))
display(HTML('<div class="irf-grid" style="grid-template-columns:repeat(3,minmax(0,330px))">' + "".join(celdas) + "</div>"))

## 7. Resumen

In [10]:
res = pd.concat(resultados, ignore_index=True)
res["robustez"] = res.get("robustez", pd.Series(index=res.index, dtype=object)).fillna("")
res["sig90"] = res.t_dk.abs() > 1.645
res["sig90_neg"] = res.sig90 & (res.beta < 0)
res["sig90_pos"] = res.sig90 & (res.beta > 0)
etiqueta_choque = {"sorpresa_tpu_std": "sorpresa TPU", "d12_tpu_std": "Δ12 TPU", "sorpresa_tpu_oos_std": "sorpresa TPU (fuera de muestra)",
                   "d12_wtui_usa_std": "WTUI EE.UU.", "d12_wtui_mundo_std": "WTUI mundo", "d12_wtui_mex_std": "WTUI México", "d12_epu_std": "EPU México"}
resumen = (res.assign(choque=res.choque.map(etiqueta_choque), variable=res.variable.map(NOMBRE_Y))
           .groupby(["especificacion", "choque", "variable", "muestra"], sort=False)
           .agg(efecto_medio_h0_12=("efecto_pp", "mean"), efecto_min=("efecto_pp", "min"), efecto_max=("efecto_pp", "max"),
                horizontes_sig90_negativos=("sig90_neg", "sum"), horizontes_sig90_positivos=("sig90_pos", "sum"),
                meses_h0=("meses", "first")).reset_index())
tabla(resumen, "Resumen por especificación, choque, variable y muestra (efectos en puntos porcentuales; de 13 horizontes)", dec=2, max_filas=200)
res.to_csv(RAIZ / "outputs/tables/lp_resultados.csv", index=False)
resumen.to_csv(RAIZ / "outputs/tables/lp_resumen.csv", index=False)
pd.DataFrame({"sorpresa_tpu": sorpresa, "sorpresa_tpu_oos": sorpresa_oos}).rename_axis("mes").to_csv(RAIZ / "outputs/tables/sorpresa_tpu.csv")
display(Markdown(f"Guardado en `outputs/tables/lp_resultados.csv` (todos los coeficientes), `lp_resumen.csv` y `sorpresa_tpu.csv`.  \n"
                 f"**Fin:** {datetime.now():%Y-%m-%d %H:%M:%S} · Python {platform.python_version()}, pandas {pd.__version__}, statsmodels {statsmodels.__version__}"))

**Resumen por especificación, choque, variable y muestra (efectos en puntos porcentuales; de 13 horizontes)**

especificacion,choque,variable,muestra,efecto_medio_h0_12,efecto_min,efecto_max,horizontes_sig90_negativos,horizontes_sig90_positivos,meses_h0
A,sorpresa TPU,Producción real,completa,0.62,-0.14,1.34,0,4,88
A,sorpresa TPU,Producción real,sin pandemia,-0.16,-1.14,0.59,4,0,72
A,sorpresa TPU,Empleo,completa,-0.04,-0.29,0.18,2,1,88
A,sorpresa TPU,Empleo,sin pandemia,0.01,-0.15,0.21,0,1,72
A,sorpresa TPU,Horas trabajadas,completa,-0.14,-0.50,0.28,3,1,88
A,sorpresa TPU,Horas trabajadas,sin pandemia,-0.10,-0.46,0.33,3,2,72
A,Δ12 TPU,Producción real,completa,0.48,-0.09,1.65,0,2,88
A,Δ12 TPU,Producción real,sin pandemia,-0.42,-1.77,1.05,3,1,72
A,Δ12 TPU,Empleo,completa,-0.08,-0.51,0.32,3,1,88
A,Δ12 TPU,Empleo,sin pandemia,0.11,-0.25,0.53,0,3,72


Guardado en `outputs/tables/lp_resultados.csv` (todos los coeficientes), `lp_resumen.csv` y `sorpresa_tpu.csv`.  
**Fin:** 2026-10-01 21:45:11 · Python 3.11.15, pandas 3.0.6, statsmodels 0.15.0